In [2]:
import os
import json
import openai
import tiktoken
import threading
import time
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

In [3]:
modelo = "gpt-4o"
# quantidade_tokens_aceitos = 4096
datasets = pd.read_csv('./dataset/dataset_frases.csv', engine='c', sep=',')
linha_inicial = 0
datasets

,ID,sentence,filtered_sentence,sentiment
0,PUtk3ryShOo,Quick Share makes the sharing process easy and...,Quick Share makes sharing process easy painless .,1
1,9LMr5XTgeyI,"But, remember, that in ye olden days of the 60...",", remember , ye olden days 60s early 70s movie...",1
2,16gB2BDXwTo,Doesn’t matter because this part didn’t really...,’ matter part ’ really go planned exactly .,2
3,TCxoZlFqzwA,- This is all color chemistry.,- color chemistry .,0
4,OFRjZtYs3wY,"Because of the way the web \nis built, sites c...","way web built , sites include material servers .",2
...,...,...,...,...
49079,A6K2dqCoin8,WE DID EXPECT THE ASTEROID TO LOOK SANDY AND B...,EXPECT ASTEROID LOOK SANDY BEACH-LIKE BASED TH...,3
49080,AqqaYs7LjlM,This means a crew could acquire a contact and ...,means crew could acquire contact lose acquire ...,3
49081,3WAkPJ6654o,"After spending the day with Dream, I've come t...","spending day Dream , 've come understand overw...",3
49082,0rkTgPt3M4k,So it's gallium nitride.,s gallium nitride .,3


In [5]:
load_dotenv()
cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
executando = True
# registro = []

In [6]:
def analisador_sentimentos(entrada):
    prompt_sistema = f"""
        You are a sentence sentiment analyzer.
        You will receive *100 separate sentences* by "||" and must return the result as a list of *exactly 100* values without spaces, where:

        1 means the sentiment is positive or good.
        0 means the sentiment is negative or bad.
        2 means the sentiment is is neutral or ivalid values.

        Do not provide any additional explanation, just return the list.

        # Format of input format

        I love this! || This is terrible. || What a great day! || I hate this.

        # Format of output format

        [1,0,1,0]

        """
    
    prompt_usuario = entrada

    lista_mensagens = [
        {
            "role": "system",
            "content": prompt_sistema
        },
        {
            "role": "user",
            "content": prompt_usuario
        }
    ]

    try:
        resposta = cliente.chat.completions.create(
            messages = lista_mensagens,
            model=modelo,
            temperature=0
        )
        texto_resposta = resposta.choices[0].message.content
    except openai.APIError as e:
        print(f"Erro de API: {e}")
    
    return texto_resposta, resposta

In [7]:
def contador_tokens(frases):
    codificador = tiktoken.encoding_for_model(modelo)
    lista_tokens = codificador.encode(frases)
    # print(f"Esse conjunto de frase tem {len(lista_tokens)} tokens")
    # print(f"Custo para o modelo {modelo} é de ${(len(lista_tokens)/1000) * 0.005}")
    return len(lista_tokens)

In [8]:
linha_atual = linha_inicial
contador = 0

In [9]:
datasets

,ID,sentence,filtered_sentence,sentiment
0,PUtk3ryShOo,Quick Share makes the sharing process easy and...,Quick Share makes sharing process easy painless .,1
1,9LMr5XTgeyI,"But, remember, that in ye olden days of the 60...",", remember , ye olden days 60s early 70s movie...",1
2,16gB2BDXwTo,Doesn’t matter because this part didn’t really...,’ matter part ’ really go planned exactly .,2
3,TCxoZlFqzwA,- This is all color chemistry.,- color chemistry .,0
4,OFRjZtYs3wY,"Because of the way the web \nis built, sites c...","way web built , sites include material servers .",2
...,...,...,...,...
49079,A6K2dqCoin8,WE DID EXPECT THE ASTEROID TO LOOK SANDY AND B...,EXPECT ASTEROID LOOK SANDY BEACH-LIKE BASED TH...,3
49080,AqqaYs7LjlM,This means a crew could acquire a contact and ...,means crew could acquire contact lose acquire ...,3
49081,3WAkPJ6654o,"After spending the day with Dream, I've come t...","spending day Dream , 've come understand overw...",3
49082,0rkTgPt3M4k,So it's gallium nitride.,s gallium nitride .,3


In [10]:
linha_inicial =  27405

In [26]:

for _ in range(10):
    datasets.to_csv('./dataset/dataset_frases.csv', index=False)
    contador+=1
    entrada = ""
    for i in range(100):
        try:
            frase = str(datasets['sentence'][linha_atual])
            if len(frase) < 200:
                entrada += "'" + datasets['sentence'][linha_atual] + "'||"
            else:
                datasets.drop(index=linha_atual)
                i -= 1
        except:
            print(f"erro linha: {linha_atual}")
            datasets.drop(index=linha_atual)
        linha_atual = linha_inicial + i 

    print(f"Efetuando a {contador} requisição...")
    # print(str)
    resposta, completo = analisador_sentimentos(entrada)
    print(f"Requisição {contador} completa")
    try:
        lista_real = json.loads(resposta)
        print(len(lista_real))
        datasets.loc[(linha_inicial):(linha_inicial+len(lista_real)-1), 'sentiment'] = lista_real
    except:
        print(f"erro Json: {completo}, efeturando linha novamente")
        contador -= 1
        linha_atual = linha_inicial

    linha_inicial = linha_atual + 1

    print(f"linha final: {linha_atual}")

erro linha: 33259
Efetuando a 59 requisição...
Requisição 59 completa
100
linha final: 33302
Efetuando a 60 requisição...
Requisição 60 completa
100
linha final: 33402
Efetuando a 61 requisição...
Requisição 61 completa
112
linha final: 33502
Efetuando a 62 requisição...
Requisição 62 completa
100
linha final: 33602
Efetuando a 63 requisição...
Requisição 63 completa
100
linha final: 33702
Efetuando a 64 requisição...
Requisição 64 completa
100
linha final: 33802
Efetuando a 65 requisição...
Requisição 65 completa
100
linha final: 33902
Efetuando a 66 requisição...
Requisição 66 completa
100
linha final: 34002
Efetuando a 67 requisição...
Requisição 67 completa
100
linha final: 34101
Efetuando a 68 requisição...
Requisição 68 completa
100
linha final: 34201


In [34]:
print(linha_inicial, linha_atual)

1782 1782


In [43]:
resposta
lista = json.loads(resposta)


In [45]:
datasets.loc[(linha_inicial):(linha_inicial+len(lista)-1), 'sentiment'] = lista

In [25]:
print(linha_inicial, len(lista))

1584 106


In [27]:
datasets['sentiment'].value_counts()

sentiment
2    29494
3    14887
1     2864
0     1839
Name: count, dtype: int64

In [ ]:
print(datasets['sentiment'].value_counts(1) + datasets['sentiment'].value_counts(0))

In [11]:
print(datasets[792:1584])

               ID                                           sentence  \
792   SA8ZBJWo73E                                            - Yeah.   
793   t705r8ICkRw  Now you might notice, we\nmention Soviet rocke...   
794   SYFuA3xnkUE   Just tap it again, and you're off running again.   
795   TCxoZlFqzwA     Thank you so much for\nsupporting the sponsor.   
796   TCxoZlFqzwA                   - So this is what it looks like.   
...           ...                                                ...   
1579  MYBef-Ggm8c  At the very least, it actually is unbelievably...   
1580  S2xHZPH5Sng  This is something all the\nbig YouTubers are d...   
1581  PUtk3ryShOo  And, finally, for better virtual collaboration...   
1582  ey_EjSzKFWQ                                    There's a glob.   
1583  094y1Z2wpJg       In fact, it climbs all\nthe way up to 9,232.   

                                      filtered_sentence  sentiment  
792                                            - Yeah .          1

In [28]:
# registro.append(f"""
#     ----------------------------------------
#     Finalizado na linha: {linha_atual}.
#     ----------------------------------------
#     """)

# with open("registro.txt", "a") as arquivo:
#     for linha in registro:
#         arquivo.write(linha + "\n")
# print("As informações foram salvas no arquivo 'registro.txt'.")

datasets.to_csv('./dataset/dataset_frases_backup.csv', index=False)
datasets.to_csv('./dataset/dataset_frases.csv', index=False)   